In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
# dbutils.widgets.text(
#     "source_folder",
#     "customers"
# )

# dbutils.widgets.text(
#     "bronze_table",
#     "bronze_inc_customers"
# )

# dbutils.widgets.text(
#     "silver_table",
#     "silver_customers"
# )

In [0]:
dbutils.widgets.dropdown(
    "dataset",
    "customers",
    ["customers", "products", "sales", "returns"]
)

dataset = dbutils.widgets.get("dataset")

print("Selected dataset:", dataset)

Selected dataset: products


In [0]:
BASE_PATH = "/Volumes/adventure_works_cata/default/adventurework_files/"

print(BASE_PATH)

/Volumes/adventure_works_cata/default/adventurework_files/


In [0]:
config = {
    "customers": {
        "target_table": "bronze_inc_customers",
        "merge_condition": "target.CustomerKey = source.CustomerKey"
    },

    "products": {
        "target_table": "bronze_inc_products",
        "merge_condition": "target.ProductKey = source.ProductKey"
    },

    "sales": {
        "target_table": "bronze_inc_sales",
        "merge_condition": """
            target.OrderNumber = source.OrderNumber
            AND target.OrderLineItem = source.OrderLineItem
        """
    },

    "returns": {
        "target_table": "bronze_inc_returns",
        "merge_condition": """
            target.ReturnDate = source.ReturnDate
            AND target.TerritoryKey = source.TerritoryKey
            AND target.ProductKey = source.ProductKey
        """
    }
}

print(config[dataset])

{'target_table': 'bronze_inc_customers', 'merge_condition': 'target.CustomerKey = source.CustomerKey'}


In [0]:
def incremental_merge(folder_name, target_table, merge_condition):

    source_path = f"{BASE_PATH}incremental/{folder_name}/"
    schema_path = f"{BASE_PATH}schemas/{folder_name}/"
    checkpoint_path = f"{BASE_PATH}checkpoints/{folder_name}/"

    print("Source:", source_path)
    print("Target:", target_table)

    # Read new files using Auto Loader
    stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", "true")
        .option("inferSchema", "true")
        .load(source_path)
        .withColumn("ingestion_time", current_timestamp())
    )

    # Process each micro-batch
    def merge_batch(batch_df, batch_id):

        if batch_df.limit(1).count() == 0:
            return

        # First run: create the Bronze table
        if not spark.catalog.tableExists(target_table):

            (
                batch_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target_table)
            )

            print(f"Created {target_table}")

        # Later runs: MERGE / UPSERT
        else:

            target = DeltaTable.forName(
                spark,
                target_table
            )

            (
                target.alias("target")
                .merge(
                    batch_df.alias("source"),
                    merge_condition
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

            print(
                f"Batch {batch_id} merged into {target_table}"
            )

    # Start Auto Loader
    query = (
        stream_df.writeStream
        .foreachBatch(merge_batch)
        .option(
            "checkpointLocation",
            checkpoint_path
        )
        .trigger(availableNow=True)
        .start()
    )

    query.awaitTermination()

    print(
        f"Incremental load completed: {target_table}"
    )

In [0]:
#creating checkpoints

In [0]:
from delta.tables import DeltaTable
incremental_merge(

    dataset,
    config[dataset]["target_table"],
    config[dataset]["merge_condition"]
)

Source: /Volumes/adventure_works_cata/default/adventurework_files/incremental/products/
Target: bronze_inc_products
Incremental load completed: bronze_inc_products


In [0]:
display(
    spark.table(
        config[dataset]["target_table"]
    )
)

CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner,_rescued_data
11000,MR.,JON,YANG,4/8/1966,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y,null
11001,MR.,EUGENE,HUANG,5/14/1965,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N,null
11002,MR.,RUBEN,TORRES,8/12/1965,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y,null
11003,MS.,CHRISTY,ZHU,2/15/1968,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N,null
11004,MRS.,ELIZABETH,JOHNSON,8/8/1968,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y,null
11005,MR.,JULIO,RUIZ,8/5/1965,S,M,julio1@adventure-works.com,"$70,000",0,Bachelors,Professional,Y,null
11007,MR.,MARCO,MEHTA,5/9/1964,M,M,marco14@adventure-works.com,"$60,000",3,Bachelors,Professional,Y,null
11008,MRS.,ROBIN,VERHOFF,7/7/1964,S,F,rob4@adventure-works.com,"$60,000",4,Bachelors,Professional,Y,null
11009,MR.,SHANNON,CARLSON,4/1/1964,S,M,shannon38@adventure-works.com,"$70,000",0,Bachelors,Professional,N,null
11010,MS.,JACQUELYN,SUAREZ,2/6/1964,S,F,jacquelyn20@adventure-works.com,"$70,000",0,Bachelors,Professional,N,null


#auto loader